# SCFA analyses — Colombian cohort


Mann–Whitney comparisons. **Supplementary Table S4

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

PROJECT = Path(os.environ.get('COLOMBIA_PROJECT_ROOT', '/content/drive/MyDrive/project'))
BASE = PROJECT / 'picrust2'
SCFA_FILE = BASE / 'inputs' / 'New SCFA Walter.xlsx'
METADATA_FILE = BASE / 'Colombian-metadata.xlsx'
OUTPUT = BASE / 'SCFA_analysis'
OUTPUT.mkdir(parents=True, exist_ok=True)
SCFA_COLS = ['Acetic', 'Propionic', 'Butyric', 'Isobutyric']

for file in (SCFA_FILE, METADATA_FILE):
    if not file.is_file():
        raise FileNotFoundError(file)

scfa = pd.read_excel(SCFA_FILE)
clinical = pd.read_excel(METADATA_FILE, sheet_name='Clinical-data')
meta = pd.read_excel(METADATA_FILE, sheet_name='Meta-data')
for table in (scfa, clinical, meta):
    table.columns = table.columns.astype(str).str.strip()

scfa['Code'] = scfa['Code'].astype(str).str.strip().str.upper()
clinical['Participant_id'] = clinical['Participant_id'].astype(str).str.strip().str.upper()
meta['Participant_id'] = meta['Participant_id'].astype(str).str.strip().str.upper()

# Verify uniqueness before joining the clinical and metabolite data.
assert scfa['Code'].is_unique
assert clinical['Participant_id'].is_unique
assert meta['Participant_id'].is_unique

merged = (scfa.merge(clinical, left_on='Code', right_on='Participant_id',
                     how='left', validate='one_to_one')
              .merge(meta[['Participant_id', 'Run']], on='Participant_id',
                     how='left', validate='one_to_one'))
merged['Depression_binary'] = (merged['Zung_S'] >= 50).astype(int)
merged['Group'] = np.where(merged['Depression_binary'] == 1, 'Depressive', 'Control')

assert len(merged) == 87
assert len(clinical) == 88
assert merged['Participant_id'].notna().all()
assert merged[['Zung_S', 'Age', 'Run'] + SCFA_COLS].notna().all().all()
assert (merged['Depression_binary'] == 1).sum() == 15
assert (merged['Depression_binary'] == 0).sum() == 72
assert (set(scfa['Code']) - set(clinical['Participant_id'])) == set()

merged_path = OUTPUT / 'Colombia_SCFA_clinical_merged_87.csv'
merged.to_csv(merged_path, index=False)

# Two-sided group comparisons with BH correction across four SCFAs.
records = []
for metabolite in SCFA_COLS:
    control = merged.loc[merged['Depression_binary'] == 0, metabolite].dropna().astype(float)
    depressive = merged.loc[merged['Depression_binary'] == 1, metabolite].dropna().astype(float)
    U, p = mannwhitneyu(depressive, control, alternative='two-sided')
    rb = (2 * U / (len(depressive) * len(control))) - 1
    direction = 'Depressive higher' if rb > 0 else ('Control higher' if rb < 0 else 'No direction')
    records.append({
        'SCFA': metabolite,
        'N_Control': len(control),
        'N_Depressive': len(depressive),
        'Control_median': control.median(),
        'Control_Q1': control.quantile(0.25),
        'Control_Q3': control.quantile(0.75),
        'Depressive_median': depressive.median(),
        'Depressive_Q1': depressive.quantile(0.25),
        'Depressive_Q3': depressive.quantile(0.75),
        'Mann_Whitney_U': U,
        'P_value': p,
        'Rank_biserial': rb,
        'Direction': direction,
    })

results = pd.DataFrame(records)
results['FDR_q'] = multipletests(results['P_value'], alpha=0.05, method='fdr_bh')[1]
results['FDR_significant'] = results['FDR_q'] < 0.05
results = results.sort_values('P_value').reset_index(drop=True)
results.to_csv(OUTPUT / 'measured_SCFA_Control_vs_Depressive_MWU.csv', index=False)
print('SCFA measurements:', len(merged), 'participants; 15 depressive and 72 controls')
print('Missing SCFA participant:', sorted(set(clinical['Participant_id']) - set(scfa['Code'])))
print(results[['SCFA', 'P_value', 'FDR_q', 'Rank_biserial']].to_string(index=False))


## Logistic regression: Supplementary Tables S2–S3

 LR baseline and SCFA-augmented models; paired bootstrap comparisons (six sequential code cells).

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

import hashlib, json, re
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score

PROJECT = Path('/content/drive/MyDrive/project')
NEW = PROJECT / 'NEW'
RUN_ROOT = NEW / 'Revision-R2c-conf'
RUN = RUN_ROOT / '01_main' / 'MAIN_conf_Age_BMI_SQ_PA_249taxa'
PART = NEW / 'inputs' / 'participants_v3_corrected_depflag.tsv'
TAXA = NEW / 'inputs' / 'taxonomy_leaf_CLR_v2.xlsx'
DIET = NEW / 'inputs' / 'diet_new_88.csv'
SCFA = PROJECT / 'picrust2' / 'SCFA_analysis' / 'Colombia_SCFA_clinical_merged_87.csv'
OUT = PROJECT / 'picrust2' / 'SCFA_analysis' / 'SCFA_LR_Confounders_Diet_Taxa_2026'

SCFA_COLS = ['Acetic','Propionic','Butyric','Isobutyric']
CONF_COLS = ['Age','BMI','SQ_all','PA']
BASE_NAME = 'Confounders + Diet + Taxa LR (baseline)'
CONFIGS = {
    BASE_NAME: [],
    'Baseline + acetate': ['Acetic'],
    'Baseline + propionate': ['Propionic'],
    'Baseline + butyrate': ['Butyric'],
    'Baseline + isobutyrate': ['Isobutyric'],
    'Baseline + all four SCFAs': SCFA_COLS,
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def mic_key(s):
    m = re.search(r'MIC[^0-9]*(\d+)', str(s).strip(), flags=re.I)
    if m is None:
        raise ValueError(f'Cannot parse MIC participant ID: {s!r}')
    return f'MIC{int(m.group(1))}'


conf_candidates = [RUN_ROOT / 'inputs' / 'confounders_88.csv',
                   NEW / 'inputs' / 'confounders_88.csv']
CONF = next((p for p in conf_candidates if p.is_file()), None)
if CONF is None:
    raise FileNotFoundError('Missing confounders_88.csv. Look in Revision-R2c-conf/inputs or NEW/inputs.')

needed = [PART,TAXA,DIET,CONF,SCFA,RUN/'run_config.json',
          RUN/'split_membership.csv',RUN/'taxa_selection.csv',RUN/'metrics_per_split.csv']
for p in needed:
    if not p.is_file(): raise FileNotFoundError(p)
OUT.mkdir(parents=True, exist_ok=True)
print('Inputs found. New outputs folder:', OUT)
print('Confounder file:', CONF)


In [ ]:
config = json.loads((RUN/'run_config.json').read_text())
assert config['n_taxa'] == 249
assert config['taxa_table'] == 'all'
assert config['confounders'] == CONF_COLS
assert tuple(config['seeds']) == (42,43,44,45,46)
expected_diet = ['Prot_pctE','SatFat_pctE','MUF_pctE','Poly_InsFat_pctE',
                 'Chol_mg_per_1000kcal','Carb_pctE','Fiber_g_per_1000kcal']
assert list(config['diet_model_inputs']) == expected_diet

for name,p in [('PART',PART),('TAXA',TAXA),('DIET',DIET),('CONFOUNDERS',CONF)]:
    expected = config['inputs'].get(name,{}).get('sha256')
    if expected:
        assert sha256(p)==expected, f"{name}: SHA-256 differs from Ali's saved main run"
    else:
        print(f'NOTE: no {name} SHA-256 recorded in run_config.json; alignment will be checked.')

part = pd.read_csv(PART, sep='\t')
diet = pd.read_csv(DIET)
taxa = pd.read_excel(TAXA, dtype={'rankID':str})
conf = pd.read_csv(CONF)
sel = pd.read_csv(RUN/'taxa_selection.csv',dtype={'rankID':str})
member = pd.read_csv(RUN/'split_membership.csv')
ali_metrics = pd.read_csv(RUN/'metrics_per_split.csv')

ids = part['participant_id'].astype(str).tolist()
assert len(ids)==len(set(ids))==88
assert diet['participant_id'].astype(str).tolist()==ids
assert conf['participant_id'].astype(str).tolist()==ids
assert taxa['rankID'].astype(str).is_unique and len(taxa)==249
sample_cols = [c for c in taxa.columns if str(c).startswith('c16S')]
assert [str(c).replace('c16S','sub-',1) for c in sample_cols]==ids
rank_ids = taxa['rankID'].astype(str).tolist()
X_taxa = taxa[sample_cols].to_numpy(dtype=float).T
X_conf = conf[CONF_COLS].to_numpy(dtype=float)
y = part['dep_flag_corrected'].to_numpy(dtype=int)
assert len(y)==88 and y.sum()==15
assert np.array_equal(y,(part['Zung_S'].astype(float).to_numpy()>=50).astype(int))
assert np.isfinite(X_taxa).all() and np.isfinite(X_conf).all()


kcal = diet['Kcal'].to_numpy(dtype=float)
assert np.isfinite(kcal).all() and (kcal>0).all()
design = pd.DataFrame(index=ids)
for raw,name,energy_per_g in [('Prot','Prot_pctE',4),('SatFat','SatFat_pctE',9),
                              ('MUF','MUF_pctE',9),('Poly_InsFat','Poly_InsFat_pctE',9),
                              ('Carb','Carb_pctE',4)]:
    design[name] = 100.0*energy_per_g*diet[raw].to_numpy(dtype=float)/kcal
design['Chol_mg_per_1000kcal'] = 1000.0*diet['Chol'].to_numpy(dtype=float)/kcal
design['Fiber_g_per_1000kcal'] = 1000.0*diet['Fiber'].to_numpy(dtype=float)/kcal
X_diet = design[expected_diet].to_numpy(dtype=float)
assert X_diet.shape==(88,7) and np.isfinite(X_diet).all()

print('Cohort: 88, depressive:',int(y.sum()),'control:',int((y==0).sum()))
print('Inputs: 7 diet + 249 candidate CLR taxa + 4 confounders')


In [ ]:
raw = pd.read_csv(SCFA)
assert len(raw)==87
assert raw['Code'].astype(str).map(mic_key).is_unique
assert (raw['Code'].astype(str).map(mic_key)==raw['Participant_id'].astype(str).map(mic_key)).all()
assert set(raw['Depression_binary'].unique()).issubset({0,1})
assert (raw['Depression_binary']==(raw['Zung_S']>=50).astype(int)).all()

source = raw.set_index(raw['Code'].map(mic_key))
participant_keys = pd.Series([mic_key(pid) for pid in ids],index=ids)
assert set(source.index).issubset(set(participant_keys))
X_scfa_df = pd.DataFrame(index=ids)
for name in SCFA_COLS:
    X_scfa_df[name] = pd.to_numeric(participant_keys.map(source[name]),errors='raise')
assert X_scfa_df.notna().sum().tolist()==[87]*4
assert np.isfinite(X_scfa_df.dropna().to_numpy()).all()
assert (X_scfa_df.dropna()>0).all().all()
missing = X_scfa_df.index[X_scfa_df.isna().any(axis=1)].tolist()
assert len(missing)==1 and mic_key(missing[0])=='MIC9'


pos = {mic_key(pid):i for i,pid in enumerate(ids)}
for row in raw.itertuples(index=False):
    j=pos[mic_key(row.Code)]
    assert int(row.Depression_binary)==int(y[j])
    assert int(row.Age)==int(X_conf[j,CONF_COLS.index('Age')])
    assert np.isclose(float(row.Zung_S),float(part.loc[j,'Zung_S']),rtol=0,atol=0.051)

qc = X_scfa_df.describe().T
qc['skewness'] = X_scfa_df.skew()
print('SCFA availability:',X_scfa_df.notna().sum().to_dict())
print('Missing ID:', missing)
print('NOTE: SCFA measurement units are not specified in this CSV; confirm from the laboratory protocol.')
display(qc.round(5))


In [ ]:
SEEDS = (42,43,44,45,46)
sel['Consensus'] = pd.to_numeric(sel['Consensus'], errors='raise').astype(int)
rows, predictions, imputations = [], [], []

for split,seed in enumerate(SEEDS,1):
    block = member[member['Split']==split]
    assert len(block)==88 and block['row_index'].is_unique
    for row in block.itertuples(index=False):
        assert ids[int(row.row_index)]==row.participant_id
        assert int(y[int(row.row_index)])==int(row.y)
    assert block['role'].value_counts().to_dict()=={'FS':36,'train':34,'test':18}
    train = block.loc[block['role']=='train','row_index'].to_numpy(dtype=int)
    test = block.loc[block['role']=='test','row_index'].to_numpy(dtype=int)
    assert y[test].sum()==3 and len(test)==18
    chosen = sel.loc[(sel['Split']==split)&(sel['Consensus']==1),'rankID'].tolist()
    assert len(chosen)==(21,16,17,16,19)[split-1]
    rank_to_index = {rank:i for i,rank in enumerate(rank_ids)}
    assert len(set(chosen))==len(chosen) and set(chosen).issubset(rank_to_index)

    taxa_idx = [i for i,rank in enumerate(rank_ids) if rank in set(chosen)]
    baseline_matrix = np.column_stack([X_diet,X_taxa[:,taxa_idx],X_conf])

    for config_name,add_cols in CONFIGS.items():
        if add_cols:
            scfa_selected = X_scfa_df[add_cols].to_numpy(dtype=float).copy()
            for j,name in enumerate(add_cols):
                med = float(np.nanmedian(scfa_selected[train,j]))
                assert np.isfinite(med)
                miss = np.isnan(scfa_selected[:,j])
                imputations.append({'Split':split,'SCFA':name,'training_median':med,
                                    'missing_in_train':int(miss[train].sum()),
                                    'missing_in_test':int(miss[test].sum())})
                scfa_selected[miss,j]=med
            X = np.column_stack([baseline_matrix,scfa_selected])
        else:
            X=baseline_matrix

        pipeline = Pipeline([
            ('scaler',StandardScaler()),
            ('clf',LogisticRegression(C=1.0,max_iter=1000,
                                      class_weight='balanced',random_state=42))
        ])
        pipeline.fit(X[train], y[train])
        score = pipeline.decision_function(X[test])
        pred = pipeline.predict(X[test])
        rows.append({'Split':split,'seed':seed,'Model':config_name,'n_taxa':len(taxa_idx),
                     'n_model_features':X.shape[1],'ROC-AUC':roc_auc_score(y[test],score),
                     'BAL-ACC':balanced_accuracy_score(y[test],pred),
                     'SEN':recall_score(y[test],pred,pos_label=1,zero_division=0)})
        for k,index in enumerate(test):
            predictions.append({'Split':split,'Model':config_name,
                                'test_position':k,'participant_id':ids[index],
                                'y_true':int(y[index]),'score':float(score[k]),
                                'y_pred':int(pred[k])})
    print('Split',split,'complete; taxa selected:',len(taxa_idx))

per_split = pd.DataFrame(rows)
pred_frame = pd.DataFrame(predictions)


ali = ali_metrics.loc[(ali_metrics['Input features']=='Confounders + diet + taxa')&
                      (ali_metrics['Model']=='LR')].sort_values('Split')
base = per_split.loc[per_split['Model']==BASE_NAME].sort_values('Split')
assert len(ali)==len(base)==5 and ali['Split'].tolist()==base['Split'].tolist()
for metric in ('ROC-AUC','BAL-ACC','SEN'):
    delta = np.abs(ali[metric].to_numpy()-base[metric].to_numpy())
    print(metric,'maximum difference:',delta.max())
    assert delta.max()<1e-6, ("Baseline does not reproduce Ali's final LR. ",
                              f'Check the inputs / package versions: {metric}')
print('PASS: Confounders + Diet + Taxa LR baseline reproduced in all five splits')
print('Baseline mean AUC:',round(base['ROC-AUC'].mean(),6))
assert abs(base['ROC-AUC'].mean()-0.596)<0.002


In [ ]:
metrics = ['ROC-AUC','BAL-ACC','SEN']
summary = per_split.groupby('Model',sort=False)[metrics].agg(['mean','std'])
summary.columns = [f'{metric}_{stat}' for metric,stat in summary.columns]
summary = summary.reset_index()
mean_base = float(base['ROC-AUC'].mean())
summary['delta_mean_AUC_vs_baseline'] = summary['ROC-AUC_mean']-mean_base

B=2000
rng=np.random.default_rng(2026)
configs=list(CONFIGS)
kernels={}
for split in range(1,6):
    ids_reference=None
    for name in configs:
        a=pred_frame[(pred_frame['Split']==split)&(pred_frame['Model']==name)].sort_values('test_position')
        current_ids=a['participant_id'].tolist()
        if ids_reference is None: ids_reference=current_ids
        else: assert current_ids==ids_reference
        yy=a['y_true'].to_numpy(dtype=int)
        scores=a['score'].to_numpy(dtype=float)
        pos=scores[yy==1];neg=scores[yy==0]
        assert len(pos)==3 and len(neg)==15
        kernels[(split,name)] = (pos[:,None]>neg[None,:]).astype(float)+0.5*(pos[:,None]==neg[None,:])

boot=np.zeros((B,len(configs)),dtype=float)
for b in range(B):
    for split in range(1,6):
        p=rng.integers(0,3,size=3)
        n=rng.integers(0,15,size=15)
        for j,name in enumerate(configs):
            boot[b,j]+=kernels[(split,name)][np.ix_(p,n)].mean()/5

differences=[]
for j,name in enumerate(configs[1:],1):
    diffs=boot[:,j]-boot[:,0]
    lo,median,hi=np.percentile(diffs,[2.5,50,97.5])
    differences.append({'Model':name,'Observed delta mean AUC':float(summary.loc[summary['Model']==name,'delta_mean_AUC_vs_baseline'].iloc[0]),
                        'Bootstrap median delta':median,'95% CI low':lo,'95% CI high':hi,
                        'Interval includes 0':bool(lo<=0<=hi)})
delta_table=pd.DataFrame(differences)
print('Classification summary:')
display(summary.round(4))
print('Paired AUC differences (SCFA-augmented model minus baseline):')
display(delta_table.round(4))
print('INTERPRETATION: exploratory internal validation; bootstrap CIs ignore training/selection variability and test-set overlap.')


In [ ]:
per_split.to_csv(OUT/'SCFA_LR_per_split.csv',index=False)
summary.to_csv(OUT/'SCFA_LR_summary.csv',index=False)
delta_table.to_csv(OUT/'SCFA_LR_paired_auc_differences.csv',index=False)
pred_frame.to_csv(OUT/'SCFA_LR_test_predictions.csv',index=False)
pd.DataFrame(imputations).to_csv(OUT/'SCFA_LR_training_median_imputation.csv',index=False)
qc.to_csv(OUT/'SCFA_measurement_QC.csv')
manifest={
    'study':'Colombian dataset SCFA exploratory LR analysis',
    'baseline_model':'Confounders + diet + taxa / LR',
    'features':{'diet':expected_diet,'confounders':CONF_COLS,'taxa':'Ali consensus-selected taxa in each split'},
    'SCFA_columns':SCFA_COLS,
    'participants':88,'SCFA_available':87,'missing_SCFA_IDs':missing,
    'imputation':'Training-set median separately by split and SCFA feature',
    'split_seeds':list(SEEDS),
    'AUC_comparison':'paired stratified test-set bootstrap, 2000 draws; no refitting',
    'limitations':['exploratory post-hoc addition of SCFAs','sample size and only 3 depressive test participants per split',
                   'overlapping test sets','bootstrap ignores training and feature-selection variability',
                   'SCFA units to confirm from original lab protocol'],
    'input_SHA256':{k:sha256(v) for k,v in [('PART',PART),('TAXA',TAXA),('DIET',DIET),('CONFOUNDERS',CONF),('SCFA',SCFA)]},
}
(OUT/'SCFA_LR_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Files saved to:',OUT)



## Correlations: Figure 4 and Supplementary Figure S2

unadjusted and confounder-adjusted Spearman correlations.

In [ ]:
"""Measured SCFA correlations with final LR-prioritized taxa, diet and confounders."""
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata


plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

PROJECT = Path(os.environ.get('COLOMBIA_PROJECT_ROOT', '/content/drive/MyDrive/project'))
NEW = PROJECT / 'NEW'
PART = NEW / 'inputs' / 'participants_v3_corrected_depflag.tsv'
DIET = NEW / 'inputs' / 'diet_new_88.csv'
TAXA = NEW / 'inputs' / 'taxonomy_leaf_CLR_v2.xlsx'
RUN_ROOT = NEW / 'Revision-R2c-conf'
SCFA = PROJECT / 'picrust2' / 'SCFA_analysis' / 'Colombia_SCFA_clinical_merged_87.csv'
CONF_OPTIONS = [RUN_ROOT / 'inputs' / 'confounders_88.csv',
                NEW / 'inputs' / 'confounders_88.csv']
CONF = next((p for p in CONF_OPTIONS if p.is_file()), CONF_OPTIONS[0])
OUT = PROJECT / 'picrust2' / 'SCFA_analysis' / 'SCFA_LR_Correlation_Heatmaps_2026_READABLE'

for path in (PART, DIET, TAXA, SCFA, CONF):
    if not path.is_file():
        raise FileNotFoundError(f'Required file missing: {path}')
OUT.mkdir(parents=True, exist_ok=True)


LR_TAXA = {
    '0.2.10.2.2.11.26': 'Lachnospiraceae_ND3007_group',
    '0.2.10.2.2.14.28': 'Ruminiclostridium_9',
    '0.2.10.2.2.2.1': 'Catabacter',
    '0.2.10.2.2.11.24': 'Lachnospiraceae_FCS020_group',
    '0.2.17.1.6.5.1': 'Rhizobiales_unclassified',
    '0.2.10.3.1.1.6': 'Dielma',
}
DIET_COLS = [
    'Chol_mg_per_1000kcal', 'Carb_pctE', 'Prot_pctE',
    'SatFat_pctE', 'Poly_InsFat_pctE', 'Fiber_g_per_1000kcal', 'MUF_pctE',
]
CONF_COLS = ['Age', 'BMI', 'SQ_all', 'PA']
SCFA_COLS = ['Acetic', 'Propionic', 'Butyric', 'Isobutyric']
GROUPS = {
    'LR taxa (CLR)': list(LR_TAXA.values()),
    'Energy-adjusted diet': DIET_COLS,
    'Confounders': CONF_COLS,
}


def mic_key(value):
    match = re.search(r'MIC[^0-9]*(\d+)', str(value).strip(), re.IGNORECASE)
    if match is None:
        raise ValueError(f'Unrecognised participant identifier: {value!r}')
    return f'MIC{int(match.group(1))}'

participants = pd.read_csv(PART, sep='\t')
diet_raw = pd.read_csv(DIET)
conf_raw = pd.read_csv(CONF)
taxa_raw = pd.read_excel(TAXA, dtype={'rankID': str})
scfa_raw = pd.read_csv(SCFA)

ids = participants['participant_id'].astype(str).tolist()
assert len(ids) == len(set(ids)) == 88, 'Expect 88 unique original participants'
assert diet_raw['participant_id'].astype(str).tolist() == ids, 'Diet IDs/order mismatch'
assert conf_raw['participant_id'].astype(str).tolist() == ids, 'Confounder IDs/order mismatch'
assert taxa_raw['rankID'].astype(str).is_unique, 'Duplicate taxa rankIDs'
assert len(taxa_raw) == 249, 'Expected 249 taxa in the original main LR analysis'

sample_cols = [c for c in taxa_raw.columns if str(c).startswith('c16S')]
assert [str(c).replace('c16S', 'sub-', 1) for c in sample_cols] == ids, \
    'CLR sample columns do not match original participant IDs/order'

assert len(scfa_raw) == 87, 'Expected 87 measured SCFA participants'
assert scfa_raw['Code'].map(mic_key).is_unique, 'Duplicate SCFA participant IDs'
assert (scfa_raw['Code'].map(mic_key) == scfa_raw['Participant_id'].map(mic_key)).all(), \
    'SCFA Code and Participant_id columns disagree'

cohort_keys = pd.Series({pid: mic_key(pid) for pid in ids})
scfa_by_mic = scfa_raw.set_index(scfa_raw['Code'].map(mic_key))
assert set(scfa_by_mic.index).issubset(set(cohort_keys)), 'Unknown SCFA participant'
missing_keys = set(cohort_keys) - set(scfa_by_mic.index)
assert len(missing_keys) == 1, f'Expected exactly one missing SCFA participant, got {missing_keys}'
print('Participant without SCFA measurement:', ', '.join(sorted(missing_keys)))


all88 = pd.DataFrame(index=pd.Index(ids, name='participant_id'))
all88['MIC_ID'] = [mic_key(s) for s in ids]
all88['Depression_binary'] = participants['dep_flag_corrected'].to_numpy(dtype=int)
assert int(all88['Depression_binary'].sum()) == 15


rank_to_row = {str(v): k for k, v in enumerate(taxa_raw['rankID'].astype(str))}
for rank_id, label in LR_TAXA.items():
    if rank_id not in rank_to_row:
        raise KeyError(f'Missing prioritised taxon rankID: {rank_id} ({label})')
    all88[label] = taxa_raw.loc[rank_to_row[rank_id], sample_cols].to_numpy(dtype=float)


energy = diet_raw['Kcal'].astype(float).to_numpy()
assert np.all(np.isfinite(energy) & (energy > 0)), 'Invalid dietary energy'
formulae = {
    'Chol_mg_per_1000kcal': (1000, 'Chol'),
    'Carb_pctE': (400, 'Carb'),
    'Prot_pctE': (400, 'Prot'),
    'SatFat_pctE': (900, 'SatFat'),
    'Poly_InsFat_pctE': (900, 'Poly_InsFat'),
    'Fiber_g_per_1000kcal': (1000, 'Fiber'),
    'MUF_pctE': (900, 'MUF'),
}
for label, (factor, name) in formulae.items():
    all88[label] = factor * diet_raw[name].astype(float).to_numpy() / energy
for name in CONF_COLS:
    all88[name] = conf_raw[name].astype(float).to_numpy()

for name in SCFA_COLS:
    mapped = all88['MIC_ID'].map(scfa_by_mic[name])
    all88[name] = pd.to_numeric(mapped, errors='raise')


scfa_lookup = scfa_raw.set_index(scfa_raw['Code'].map(mic_key))
for _, row in all88.dropna(subset=SCFA_COLS).iterrows():
    source_row = scfa_lookup.loc[row['MIC_ID']]
    assert int(source_row['Depression_binary']) == int(row['Depression_binary']), \
        f"Depression label mismatch for {row['MIC_ID']}"
    assert np.isclose(float(source_row['Age']), float(row['Age']), atol=1e-8), \
        f"Age mismatch for {row['MIC_ID']}"


columns = list(LR_TAXA.values()) + DIET_COLS + CONF_COLS
analysis = all88.dropna(subset=SCFA_COLS + columns).copy()
assert len(analysis) == 87, f'Expected 87 complete cases; got {len(analysis)}'
assert analysis['MIC_ID'].is_unique
assert np.isfinite(analysis[SCFA_COLS + columns].to_numpy(dtype=float)).all()
assert (analysis[SCFA_COLS] > 0).all().all()
print(f'Analysis sample: {len(analysis)} unique participants, '
      f"{int(analysis['Depression_binary'].sum())} depressive, "
      f"{int((analysis['Depression_binary'] == 0).sum())} controls")


def bh_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    if not np.all(np.isfinite(p) & (p >= 0) & (p <= 1)):
        raise ValueError('Invalid p-value before BH adjustment')
    order = np.argsort(p)
    ranked = p[order]
    m = len(p)
    adjusted_ranked = np.minimum.accumulate((ranked * m / np.arange(1, m+1))[::-1])[::-1]
    adjusted = np.empty(m)
    adjusted[order] = np.minimum(adjusted_ranked, 1)
    return adjusted

records = []
for scfa in SCFA_COLS:
    for category, group_cols in GROUPS.items():
        for feature in group_cols:
            rho, p = stats.spearmanr(analysis[scfa], analysis[feature], nan_policy='raise')
            if not (np.isfinite(rho) and np.isfinite(p)):
                raise ValueError(f'Undefined Spearman correlation for {scfa} vs {feature}')
            records.append(dict(SCFA=scfa, Feature=feature, Group=category, n=len(analysis),
                                spearman_rho=float(rho), p_two_sided=float(p)))

res = pd.DataFrame(records)
assert len(res) == 4 * 17
res['q_BH_68_tests'] = bh_adjust(res['p_two_sided'])
res['FDR_q_lt_0.05'] = res['q_BH_68_tests'] < 0.05
res.to_csv(OUT / 'SCFA_Spearman_all_68_tests.csv', index=False)


conf_ranks = np.column_stack([rankdata(analysis[c].to_numpy()) for c in CONF_COLS])
Z = np.column_stack([np.ones(len(analysis)), conf_ranks])
assert np.linalg.matrix_rank(Z) == 5, 'Confounder rank matrix is singular'

def residual_ranks(series):
    ranks = rankdata(np.asarray(series, dtype=float), method='average')
    coefficients = np.linalg.lstsq(Z, ranks, rcond=None)[0]
    return ranks - Z @ coefficients

partial_records = []
df_resid = len(analysis) - len(CONF_COLS) - 2
assert df_resid > 0
for scfa in SCFA_COLS:
    ry = residual_ranks(analysis[scfa])
    for category in ('LR taxa (CLR)', 'Energy-adjusted diet'):
        for feature in GROUPS[category]:
            rx = residual_ranks(analysis[feature])
            r = float(np.corrcoef(ry, rx)[0, 1])
            if not np.isfinite(r):
                raise ValueError(f'Invalid partial r for {scfa} and {feature}')
            if abs(r) >= 1:
                p = 0.0
            else:
                tvalue = r * np.sqrt(df_resid / (1 - r*r))
                p = float(2 * stats.t.sf(abs(tvalue), df_resid))
            partial_records.append(dict(SCFA=scfa, Feature=feature, Group=category,
                                        n=len(analysis), covariates='Age; BMI; SQ_all; PA',
                                        partial_spearman_r=r, p_approx_two_sided=p))
partial = pd.DataFrame(partial_records)
assert len(partial) == 4 * 13
partial['q_BH_52_tests'] = bh_adjust(partial['p_approx_two_sided'])
partial['FDR_q_lt_0.05'] = partial['q_BH_52_tests'] < 0.05
partial.to_csv(OUT / 'SCFA_partial_Spearman_adjusted_52_tests.csv', index=False)


def make_matrix(frame, value, cols):
    return (frame.pivot(index='SCFA', columns='Feature', values=value)
            .reindex(index=SCFA_COLS, columns=cols))


def draw_heatmap(frame, r_column, q_column, grouped_columns, title, stem, q_family):
    """Render the correlation matrix as a manuscript-width vector heatmap."""
    cols = [c for groupcols in grouped_columns.values() for c in groupcols]
    rho = make_matrix(frame, r_column, cols)
    q = make_matrix(frame, q_column, cols)
    if rho.isna().any().any() or q.isna().any().any():
        raise ValueError('Heatmap has missing correlations')


    rho.to_csv(OUT / f'{stem}_rho_matrix.csv')
    q.to_csv(OUT / f'{stem}_q_matrix.csv')


    is_main = len(cols) == 17
    fig = plt.figure(figsize=(7.2, 4.15 if is_main else 4.0), facecolor='white')
    ax = fig.add_axes([0.125, 0.405, 0.754, 0.43 if is_main else 0.45])
    cax = fig.add_axes([0.900, 0.405, 0.020, 0.43 if is_main else 0.45])

    image = ax.imshow(rho.to_numpy(dtype=float), cmap='RdBu_r',
                      vmin=-1, vmax=1, interpolation='nearest', aspect='auto')


    label_map = {
        'Lachnospiraceae_ND3007_group': 'ND3007_group',
        'Lachnospiraceae_FCS020_group': 'FCS020_group',
        'Rhizobiales_unclassified': 'Rhizobiales_uncl.',
        'Chol_mg_per_1000kcal': 'Chol/1000 kcal',
        'Carb_pctE': 'Carb %E', 'Prot_pctE': 'Protein %E',
        'SatFat_pctE': 'Sat. fat %E',
        'Poly_InsFat_pctE': 'Polyunsat. fat %E',
        'Fiber_g_per_1000kcal': 'Fibre/1000 kcal',
        'MUF_pctE': 'Monounsat. fat %E',
        'SQ_all': 'SQ_all (ln LF/HF)', 'PA': 'PA (1-3)',
    }
    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels([label_map.get(c, c) for c in cols],
                       rotation=63, ha='right', rotation_mode='anchor',
                       fontsize=8.0 if is_main else 8.5)
    ax.set_yticks(np.arange(len(SCFA_COLS)))
    ax.set_yticklabels(['Acetate', 'Propionate', 'Butyrate', 'Isobutyrate'],
                       fontsize=9.5, fontweight='medium')
    ax.tick_params(which='major', length=0, pad=3)


    for i in range(len(SCFA_COLS)):
        for j in range(len(cols)):
            value = float(rho.iat[i, j])
            significant = bool(q.iat[i, j] < 0.05)
            txt = f'{value:+.2f}' + ('*' if significant else '')
            ax.text(j, i, txt, ha='center', va='center',
                    fontsize=7.3 if is_main else 8.1,
                    fontweight='bold' if significant else 'medium',
                    color='white' if abs(value) > 0.58 else '#1b2632')


    start_col = 0
    for group_name, groupcols in grouped_columns.items():
        n = len(groupcols)
        if start_col:
            ax.axvline(start_col - 0.5, color='#222222', linewidth=1.2)
        center = (start_col + n / 2) / len(cols)
        display_name = {'LR taxa (CLR)': 'Taxa', 'Energy-adjusted diet': 'Diet'}.get(group_name, group_name)
        ax.text(center, 1.045, display_name, ha='center', va='bottom',
                transform=ax.transAxes, fontsize=8.2,
                fontweight='bold', clip_on=False)
        start_col += n

    ax.set_xlim(-0.5, len(cols) - 0.5)
    ax.set_ylim(len(SCFA_COLS) - 0.5, -0.5)
    ax.set_xticks(np.arange(-0.5, len(cols), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(SCFA_COLS), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.9)
    ax.tick_params(which='minor', bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_edgecolor('#515b66')

    cbar = fig.colorbar(image, cax=cax)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.ax.tick_params(labelsize=7.4, length=2, pad=2)
    cbar.set_label('Correlation coefficient', fontsize=8.5, labelpad=5)


    fig.text(0.104, 0.960, title, ha='left', va='top', fontsize=10.0,
             fontweight='bold', color='#19232d')
    fig.text(0.104, 0.055,
             f'n = {len(analysis)}; * BH-FDR q < 0.05 across {q_family} tests; '
             'exploratory, non-causal associations.',
             fontsize=8.0, color='#434c57', ha='left', va='bottom')


    for ext in ('pdf', 'png'):
        fig.savefig(OUT / f'{stem}.{ext}', dpi=400,
                    facecolor='white', bbox_inches=None)
    plt.show()
    plt.close(fig)


draw_heatmap(
    res, 'spearman_rho', 'q_BH_68_tests', GROUPS,
    'SCFA correlations (unadjusted Spearman)',
    'SCFA_Spearman_4x17_heatmap', 68,
)
draw_heatmap(
    partial, 'partial_spearman_r', 'q_BH_52_tests',
    {'Taxa': list(LR_TAXA.values()), 'Diet': DIET_COLS},
    'SCFA correlations (confounder-adjusted partial Spearman)',
    'SCFA_partial_Spearman_4x13_heatmap', 52,
)


key = pd.DataFrame([{'rankID': rid, 'Taxon': label} for rid, label in LR_TAXA.items()])
key.to_csv(OUT / 'LR_prioritised_taxa_rankID_key.csv', index=False)
# The following file contains participant-level study data; do not publish it without approval.
analysis[['MIC_ID', 'Depression_binary'] + SCFA_COLS + list(LR_TAXA.values())
         + DIET_COLS + CONF_COLS].to_csv(OUT / 'aligned_87_complete_case_analysis_inputs.csv', index=False)

notes = {
    'study': 'Colombian cohort; n=87 with measured SCFA; cross-sectional',
    'source_scfa': str(SCFA),
    'taxa_source': str(TAXA),
    'diet_source': str(DIET),
    'confounder_source': str(CONF),
    'n_total_cohort': 88,
    'n_complete_scfa': len(analysis),
    'missing_scfa_participant': sorted(missing_keys),
    'scfas': SCFA_COLS,
    'taxa_rankIDs': LR_TAXA,
    'diet_features': DIET_COLS,
    'covariates': CONF_COLS,
    'spearman': 'Two-sided scipy Spearman rank correlation on 87 complete cases; BH across all 68 tests (4 SCFAs x 17 features).',
    'partial': 'Rank-residual partial Spearman adjusting jointly for four covariates (including ranked PA); approximate t-test df=n-4-2; BH across 52 tests (4 SCFAs x 13 non-confounder features).',
    'warnings': ['Cross-sectional associations are not causal.',
                 'Final taxa were outcome-prioritised; FDR does not account for this selection.',
                 'Partial Spearman p-values are approximate, and small sample size limits precision.',
                 'Correlations use 87 actual measurements: no SCFA median imputation.',
                 'SCFA units not specified by provided SCFA CSV.',
                 'Confounders may themselves associate with SCFA; these are not model performance measures.'],
}
(OUT / 'correlation_methods_and_QC.json').write_text(json.dumps(notes, indent=2), encoding='utf-8')

print('\nSpearman: significant at BH q<0.05:', int(res['FDR_q_lt_0.05'].sum()), '/ 68')
print('Partial Spearman: significant at BH q<0.05:', int(partial['FDR_q_lt_0.05'].sum()), '/ 52')
print('\nTop 10 Spearman correlations by |rho|:')
print(res.loc[res['spearman_rho'].abs().sort_values(ascending=False).index,
              ['SCFA', 'Feature', 'n', 'spearman_rho', 'p_two_sided', 'q_BH_68_tests']]
         .head(10).to_string(index=False, float_format=lambda x: f'{x:.4g}'))
print('\nSaved everything to:', OUT)


## Manuscript cross-check



In [ ]:
# Expected values are from the final manuscript, with tolerances for rounding.
expected_aucs = {
    BASE_NAME: 0.595556,
    'Baseline + acetate': 0.6000,
    'Baseline + propionate': 0.5689,
    'Baseline + butyrate': 0.5733,
    'Baseline + isobutyrate': 0.5822,
    'Baseline + all four SCFAs': 0.5511,
}
observed_aucs = summary.set_index('Model')['ROC-AUC_mean']
assert len(per_split) == 30 and len(observed_aucs) == 6
for model, expected in expected_aucs.items():
    observed = float(observed_aucs.loc[model])
    assert abs(observed - expected) < 0.0005, f'{model}: {observed:.6f} != {expected:.6f}'
assert len(delta_table) == 5 and delta_table['Interval includes 0'].all()
assert len(results) == 4 and not results['FDR_significant'].any()
assert abs(results['P_value'].min() - 0.179343) < 0.0005
assert abs(results['FDR_q'].min() - 0.717371) < 0.0005
assert len(res) == 68 and len(partial) == 52
assert int(res['FDR_q_lt_0.05'].sum()) == 1
assert int(partial['FDR_q_lt_0.05'].sum()) == 0
main_corr = res[(res['SCFA'] == 'Acetic') & (res['Feature'] == 'Rhizobiales_unclassified')].iloc[0]
assert abs(float(main_corr['spearman_rho']) - 0.372) < 0.001
assert abs(float(main_corr['q_BH_68_tests']) - 0.027) < 0.001
adjusted_corr = partial[(partial['SCFA'] == 'Acetic') &
                        (partial['Feature'] == 'Rhizobiales_unclassified')].iloc[0]
assert abs(float(adjusted_corr['partial_spearman_r']) - 0.282) < 0.003
assert abs(float(adjusted_corr['p_approx_two_sided']) - 0.010) < 0.003
assert abs(float(adjusted_corr['q_BH_52_tests']) - 0.257) < 0.003
print('PASS: SCFA S2–S4, Figure 4, and Supplementary Figure S2 match the manuscript.')
print('Caution: this confirms numerical reproduction, not statistical validity or data-release permissions.')